# Занятие 5 — Исследовательский анализ данных

## Подготовка

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = Path("../data")
WORK = Path("../work")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(DATA / "decisions.csv",dtype={"lei":"string","county_code":"string"})
assert len(decisions)==6830 and decisions["source_row"].is_unique
assert decisions["loan_amount"].notna().all() and decisions["loan_amount"].gt(0).all()


## B03 — слайд 3: Проверяем единицы и известные значения

In [ ]:
eda = decisions.copy()
eda["loan_kusd"] = eda["loan_amount"] / 1000
audit = pd.DataFrame({"rows":len(eda), "known":eda[["loan_amount","income"]].count()})
audit["missing"] = audit["rows"] - audit["known"]
print(audit)
print("Доход <= 0:", eda["income"].le(0).sum())
assert audit.loc["income","known"] == 6488

## B04 — слайд 4: Среднее: сумма, делённая на количество

In [ ]:
small = eda.head(4)["loan_kusd"]
print(small.tolist(), small.sum(), len(small))
print(small.sum() / len(small), small.mean())
assert small.mean() == 357.5
print("Средняя сумма, USD:", eda["loan_amount"].mean())

## B05 — слайд 5: Медиана: середина упорядоченного списка

In [ ]:
print(small.sort_values().tolist())
print(small.median(), eda["loan_amount"].median())
assert small.median() == 285

## B06 — слайд 6: Мода и частоты

In [ ]:
print(small.mode(), type(small.mode()))
print(eda["action_taken"].value_counts(dropna=False).reindex([1,2,3]))

## B07 — слайд 7: Квантили: значения на заданных местах

In [ ]:
print(small.quantile([.25,.5,.75], interpolation="linear"))
quantiles = eda["loan_amount"].quantile([.1,.25,.5,.75,.9,.99])
print(quantiles)
assert quantiles.loc[.25] == 315000 and quantiles.loc[.75] == 755000

## B08 — слайд 8: Как описать разброс

In [ ]:
loan = eda["loan_amount"]
q1,q3 = loan.quantile([.25,.75])
iqr = q3-q1
print("Диапазон, USD:", loan.max()-loan.min())
print("IQR, USD:", iqr)
print(eda.nlargest(3,"loan_amount")[["source_row","loan_amount","income"]])

## B09 — слайд 9: Стандартное отклонение

In [ ]:
deviation = small - small.mean()
squares = deviation ** 2
print(pd.DataFrame({"x":small,"deviation":deviation,"square":squares}))
print(squares.sum()/len(small), small.var(ddof=0))
print(small.std(ddof=1), np.std(small.to_numpy(),ddof=1))
assert np.isclose(small.var(ddof=1),squares.sum()/(len(small)-1))

## B10 — слайд 10: describe: сводка с понятными строками

In [ ]:
overview = eda[["loan_amount","income"]].describe(percentiles=[.1,.25,.5,.75,.9,.99]).T
overview.round(1)

## B11 — слайд 11: Гистограмма: сколько значений в интервале

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4),layout="constrained")
values = eda["loan_kusd"]
axes[0].hist(values,bins=40,color="#0072B2",edgecolor="white")
zoom = values[values.le(3000)]
axes[1].hist(zoom,bins=np.arange(0,3100,100),color="#0072B2",edgecolor="white")
for ax in axes: ax.set(xlabel="Сумма, тыс. USD",ylabel="Число записей")
axes[0].set_title(f"Полный диапазон: n={len(values)}")
axes[1].set_title(f"Увеличение <=3000: n={len(zoom)}; вне {len(values)-len(zoom)}")
plt.show()

## B12 — слайд 12: ECDF: доля не выше заданной суммы

In [ ]:
print("Доля сумм <=500000:", loan.le(500000).mean())
x = np.sort(eda["loan_kusd"].to_numpy())
y = np.arange(1,len(x)+1)/len(x)
fig,ax=plt.subplots(figsize=(9,4),layout="constrained")
ax.step(x,y,where="post",color="#0072B2")
ax.set_xscale("log",base=10)
ax.set(xlabel="Сумма, тыс. USD; log10",ylabel="Доля не выше порога",ylim=(0,1))
plt.show()

## B13 — слайд 13: Асимметрия распределения

In [ ]:
print("skew loan_amount:", loan.skew())
print("mean/median, USD:", loan.mean(),loan.median())

## B14 — слайд 14: Boxplot и правило IQR

In [ ]:
upper = q3+1.5*iqr
tail = loan.gt(upper)
print(upper,int(tail.sum()))
assert upper == 1415000 and tail.sum() == 355
fig,ax=plt.subplots(figsize=(9,3),layout="constrained")
ax.boxplot(eda["loan_kusd"],vert=False,whis=1.5)
ax.set_xscale("log",base=10)
ax.set(xlabel="Сумма, тыс. USD; log10",yticks=[])
plt.show()

## B15 — слайд 15: Что изменится при исключении флагов

In [ ]:
sensitivity = pd.DataFrame({"full":loan.describe(),"without_IQR_tail":loan[~tail].describe()}).T
sensitivity[["count","mean","50%","std"]].round(1)

## B16 — слайд 16: Сравнение групп

In [ ]:
by_outcome = eda.groupby("action_taken",dropna=False).agg(
    n=("source_row","size"), loan_median=("loan_amount","median"),
    income_known=("income","count"),income_median=("income","median"))
by_outcome
by_outcome["income_missing"] = by_outcome["n"] - by_outcome["income_known"]
by_outcome

## B17 — слайд 17: Таблица сочетаний и доли

In [ ]:
eda["income_missing"] = eda["income"].isna()
counts = pd.crosstab(index=eda["action_taken"],columns=eda["income_missing"])
shares = pd.crosstab(index=eda["action_taken"],columns=eda["income_missing"],normalize="index")
print(counts)
print(shares.round(3))
assert counts.to_numpy().sum() == 6830

## B19 — слайд 19: Среднее групп и среднее записей

In [ ]:
groups = eda.groupby("lei").agg(n=("loan_amount","count"),mean_sum=("loan_amount","mean"))
weighted = (groups["n"]*groups["mean_sum"]).sum()/groups["n"].sum()
print(groups["mean_sum"].mean(),weighted,loan.mean())
assert np.isclose(weighted,loan.mean())

## B20 — слайд 20: Связь суммы с доходом

In [ ]:
pairs = eda.dropna(subset=["loan_kusd","income"])
assert len(pairs) == 6488
fig,ax=plt.subplots(figsize=(9,4),layout="constrained")
ax.scatter(pairs["income"],pairs["loan_kusd"],s=10,alpha=.3,color="#0072B2")
ax.set(xlabel="Доход, тыс. USD/год",ylabel="Сумма, тыс. USD",title=f"Все известные пары: n={len(pairs)}")
plt.show()

## B21 — слайд 21: Корреляция Pearson

In [ ]:
numeric = eda[["loan_amount","income"]]
pearson = numeric.corr(method="pearson",min_periods=30)
print(pearson)
print("n полных пар:",numeric.dropna().shape[0])
print("Без IQR-хвоста:",numeric.loc[~tail].corr().iloc[0,1])

## B22 — слайд 22: Корреляция Spearman

In [ ]:
spearman = numeric.corr(method="spearman",min_periods=30)
print(spearman)
print("Pearson/Spearman:",pearson.iloc[0,1],spearman.iloc[0,1])